# 3.3. Synthetic Regression Data

Let's prepare some synthetic regression data for training our first linear regression model in the next chapter. Synthetic regression data is, as its name suggests, not related to any real-world data or scenario. However, a key advantage of synthetic data is that it can be generated at will for testing machine learning algorithms and techniques, without the usual difficulties and concerns of obtaining real-world data, whether technically, legally or morally.

As described in chapter 3.2, the [D2L](https://www.d2l.ai/) textbook introduces its own Python library [`d2l`](https://pypi.org/project/d2l/) containing useful classes and methods for abstracting common machine learning operations and facilitating code reuse. However, this approach has a few drawbacks.

1. The `d2l` Python library was last updated in August 2023 with no signs of further feature or security updates available
1. The `d2l` library is non-standard and seldom used in real-world machine learning contexts outside of the D2L textbook itself

Therefore, we will take a different approach here and use a mature, production-ready Python library called [scikit-learn](https://scikit-learn.org/stable/) to generate our synthetic regression data. The data is returned as Numpy arrays so we can covert them directly to MindSpore tensors and feed them into our model for training and inference.

At the time of writing \(March 2026\), the latest version of `scikit-learn` is `1.8.0`. Let's install it!

In [1]:
%pip install scikit-learn==1.8.0


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 3.3.1. Generating the Dataset

Let's generate 1000 data samples with each sample containing 2 features drawn from the standard normal distribution $N(0, 1)$. Denote the resulting matrix as $\mathbf{X}_{1000\times2}$.

Then, we generate each label by applying a _ground truth_ linear function plus some additive noise $\mathbf{\epsilon}$ drawn from a normal distribution $N(0, 0.0001)$ with mean $\mu = 0$ and standard deviation $\sigma = \sqrt{\sigma^2} = \sqrt{0.0001} = 0.01$.

$$
\begin{align}
\mathbf{y} &= \mathbf{Xw} + b + \mathbf{\epsilon}
\end{align}
$$

With the [`sklearn.datasets.make_regression`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_regression.html) method, we can specify the bias, say $b = 4.2$ and the standard deviation of the additive noise. However, we cannot specify the weight vector $\mathbf{w}$ directly. Instead, we can set the `coef` argument to `True` and the method will return the underlying weight vector $\mathbf{w}$ along with the generated features $\mathbf{X}$ and labels $\mathbf{y}$.

In [2]:
import sklearn.datasets

bias = 4.2
X, y, coef = sklearn.datasets.make_regression(n_samples=1000,
                                              n_features=2,
                                              n_targets=1,
                                              bias=bias,
                                              noise=0.01,
                                              coef=True)
X.shape, y.shape, coef.shape

((1000, 2), (1000,), (2,))

Let's look at the first sample.

In [3]:
X[0], y[0]

(array([-0.54701702, -0.48273246]), -30.956464970577134)

Let's inspect the generated weights in `coef`.

In [4]:
coef

array([22.17761777, 47.70670787])

Denote the features for our first sample as $\mathbf{x}$ and its corresponding label as $y$. Since the additive noise is minimal, we expect our label $y$ to approximately equal $\mathbf{x}^{\top}\mathbf{w} + b$. Verify if this is the case.

In [5]:
import numpy as np

y[0], np.dot(X[0], coef) + bias

(-30.956464970577134, -30.961110642110075)

Great, both values are similar as expected, almost identical.

## 3.3.2. Reading the Dataset

The machine learning pipeline broadly consists of the following high-level processes.

1. Data loading, cleaning and validation
1. Model training and fine-tuning
1. Model evaluation and export
1. Model loading and inference

Steps \(2\) and \(3\) require data for model training and evaluation respectively. Instead of obtaining the training and evaluation data from separate sources, it is common to split the data from \(1\) into separate training and test datasets. This is known as a train-test split and the functionality is exposed in scikit-learn via the method [`sklearn.model_selection.train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

In the previous section we had a synthetically generated dataset with 1000 samples including both features and labels. Let's split them based on the default options provided by scikit-learn.

1. 75% will be used as training data for a total of 750 samples
1. The remainder 25% will be used as testing data for a total of 250 samples
1. By default, scikit-learn shuffles the dataset before splitting, preserving the relationship between features and corresponding labels

In [6]:
import sklearn.model_selection

X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((750, 2), (250, 2), (750,), (250,))

Let's verify the relationship between features and labels are preserved. For example, take the first sample in the \(now shuffled\) training dataset. Notice the following.

1. It's \(very likely\) different from the first sample we saw earlier. This suggests the dataset has been shuffled
1. We still have $y \approx \mathbf{x}^{\top}\mathbf{w} + b$ assuming the additive noise is negligible. This suggests the relationship between features and labels is preserved

In [7]:
X0_train = X_train[0]
y0_train = y_train[0]

X0_train, y0_train, np.dot(X0_train, coef) + bias

(array([ 1.5858464 , -0.45610373]), 17.595711475478083, 17.611088185135777)

Another common operation on datasets is _batching_. Batching refers to the process of splitting the dataset in chunks of the same size, usually a power of 2. For example, our training dataset of 750 samples could be split into 23 batches of $2^5 = 32$, with the last batch of $750 - 23 \times 32 = 14$ remaining samples optionally excluded from the pass within the training process. This technique is useful especially when the size of the dataset is large and may not fit entirely in memory when loaded at once.

In MindSpore 2.8.0, this is done via the `batch` method provided by the [`mindspore.dataset.NumpySlicesDataset`](https://www.mindspore.cn/docs/en/r2.8.0/api_python/dataset/mindspore.dataset.NumpySlicesDataset.html) class.

In [8]:
import mindspore
mindspore.set_device(device_target='Ascend', device_id=0)

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2


In [9]:
import mindspore.dataset as ds
import mindspore.ops as ops

train_ds = ds.NumpySlicesDataset(data=(X_train, y_train), column_names=['features', 'labels'], shuffle=True)
train_ds = train_ds.batch(32, drop_remainder=True)
weights = mindspore.Tensor(coef)
for batch in train_ds.create_dict_iterator():
    X_batch = batch['features']
    y_batch = batch['labels']
    print(f'X_batch shape: {X_batch.shape}')
    print(f'y_batch shape: {y_batch.shape}')
    x0 = X_batch[0]
    y0 = y_batch[0]
    print(f'x0 = {x0}, y0 = {y0}, dot(x0, w) + b = {ops.vecdot(x0, weights) + bias}')

X_batch shape: (32, 2)
y_batch shape: (32,)
x0 = [-0.25827668 -0.95457362], y0 = -47.079992043286175, dot(x0, w) + b = -47.06752632506634
X_batch shape: (32, 2)
y_batch shape: (32,)
x0 = [-0.54810594 -0.10743811], y0 = -13.069056405805185, dot(x0, w) + b = -13.081202319394528
X_batch shape: (32, 2)
y_batch shape: (32,)
x0 = [-1.64119791 -2.35541132], y0 = -144.58590470891994, dot(x0, w) + b = -144.56677968948446
X_batch shape: (32, 2)
y_batch shape: (32,)
x0 = [-0.38210472  1.13135441], y0 = 49.686232131878945, dot(x0, w) + b = 49.69902165856059
X_batch shape: (32, 2)
y_batch shape: (32,)
x0 = [-0.68000442  0.48782594], y0 = 12.384267661153572, dot(x0, w) + b = 12.391691823477732
X_batch shape: (32, 2)
y_batch shape: (32,)
x0 = [ 0.83356045 -0.52745699], y0 = -2.4691815215849195, dot(x0, w) + b = -2.476851344118045
X_batch shape: (32, 2)
y_batch shape: (32,)
x0 = [-0.21806441  0.26720661], y0 = 12.103288322898708, dot(x0, w) + b = 12.11139825954583
X_batch shape: (32, 2)
y_batch shape:

In general, loading large datasets with `mindspore.dataset.NumpySlicesDataset` directly is not recommended since the entire dataset is loaded to memory at once, defeating the purpose of batching. MindSpore provides a more generic [`mindspore.dataset.GeneratorDataset`](https://www.mindspore.cn/docs/en/r2.8.0/api_python/dataset/mindspore.dataset.GeneratorDataset.html) class which allows defining a custom generator passed as the data source, allowing the data to be loaded on demand in a lazy manner.

## 3.3.4. Summary

Now that we know how to generate, load, split and batch our data for machine learning algorithms, let's train our first linear regression model in the next chapter!